# Part 3 — SQL Analysis

**Week 8: E-Commerce Order Analytics System**

Loads the cleaned CSV files into SQLite and performs all SQL analysis required in the assignment.

Covered:
- Basic queries
- Intermediate queries
- Running totals
- DENSE_RANK
- LAG / LEAD
- Multi-level CTEs
- NTILE segmentation
- Year-over-Year comparison
- First/Last category analysis
- Cumulative distribution
- Cohort analysis
- Self-join with window function
- Frequently bought together

In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_FILE = "ecommerce.db"

conn = sqlite3.connect(DB_FILE)

print("SQLite database connected successfully.")

SQLite database connected successfully.


## 1. Load Cleaned CSV Files into SQLite

In [3]:
orders = pd.read_csv("data/cleaned_orders.csv")
order_items = pd.read_csv("data/cleaned_order_items.csv")
products = pd.read_csv("data/cleaned_products.csv")
customers = pd.read_csv("data/cleaned_customers.csv")

orders.to_sql("orders", conn, if_exists="replace", index=False)
order_items.to_sql("order_items", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)

print("Tables created:")
print("✓ orders")
print("✓ order_items")
print("✓ products")
print("✓ customers")

Tables created:
✓ orders
✓ order_items
✓ products
✓ customers


## 2. Check Database Tables

In [5]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn
)
display(tables)

for table in ["orders", "order_items", "products", "customers"]:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS row_count FROM {table};", conn
    ).iloc[0]["row_count"]
    print(f"{table}: {count} rows")

,name
0,customers
1,order_items
2,orders
3,products


orders: 1000 rows
order_items: 2500 rows
products: 100 rows
customers: 500 rows


## Basic Queries

### 3. Total Revenue per Category

In [6]:
query = '''
SELECT
    p.category,
    ROUND(SUM(
        oi.quantity * oi.unit_price *
        (1 - oi.discount_percent / 100.0)
    ), 2) AS total_revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
JOIN orders o
    ON oi.order_id = o.order_id
WHERE o.status NOT IN ('CANCELLED')
GROUP BY p.category
ORDER BY total_revenue DESC;
'''

display(pd.read_sql_query(query, conn))

,category,total_revenue
0,Books,28894923.89
1,Electronics,25936118.89
2,Home,25181119.93
3,Clothing,23576576.62


### 4. Top 10 Customers by Total Order Value

In [7]:
query = '''
SELECT
    c.customer_id,
    c.customer_name,
    ROUND(SUM(
        oi.quantity * oi.unit_price *
        (1 - oi.discount_percent / 100.0)
    ), 2) AS total_order_value
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE o.status NOT IN ('CANCELLED')
GROUP BY c.customer_id, c.customer_name
ORDER BY total_order_value DESC
LIMIT 10;
'''

display(pd.read_sql_query(query, conn))

,customer_id,customer_name,total_order_value
0,CUST0497,Sneha Pawar,1040001.00
1,CUST0425,Anjali Joshi,1034103.69
2,CUST0148,Rohan Kulkarni,974070.73
3,CUST0372,Akash Patil,929067.63
4,CUST0308,Rohan Joshi,889125.72
5,CUST0121,Sneha Pawar,876064.72
6,CUST0289,Pooja Deshmukh,868821.49
7,CUST0196,Amit Deshmukh,866321.89
8,CUST0412,Amit Patil,838778.29
9,CUST0275,Priya Patil,797089.59


### 5. Month-wise Order Count for the Last 12 Months

In [8]:
query = '''
WITH max_date AS (
    SELECT MAX(date(order_date)) AS latest_date
    FROM orders
)
SELECT
    strftime('%Y-%m', order_date) AS order_month,
    COUNT(*) AS order_count
FROM orders, max_date
WHERE date(order_date) >= date(max_date.latest_date, '-11 months')
GROUP BY strftime('%Y-%m', order_date)
ORDER BY order_month;
'''

display(pd.read_sql_query(query, conn))

,order_month,order_count
0,2025-06,30
1,2025-07,60
2,2025-08,62
3,2025-09,63
4,2025-10,66
5,2025-11,67
6,2025-12,68
7,2026-01,66
8,2026-02,54
9,2026-03,58


## Intermediate Queries

### 6. Customers Who Ordered but Never Had a Delivered Item

In [9]:
query = '''
SELECT
    c.customer_id,
    c.customer_name
FROM customers c
WHERE EXISTS (
    SELECT 1
    FROM orders o
    WHERE o.customer_id = c.customer_id
)
AND NOT EXISTS (
    SELECT 1
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id = c.customer_id
      AND o.status = 'DELIVERED'
)
ORDER BY c.customer_id;
'''

display(pd.read_sql_query(query, conn))

,customer_id,customer_name
0,CUST0001,Amit Sharma
1,CUST0004,Rahul Sharma
2,CUST0005,Sneha Sharma
3,CUST0007,Anjali Deshmukh
4,CUST0008,Priya More
...,...,...
285,CUST0490,Akash Joshi
286,CUST0491,Neha Jadhav
287,CUST0494,Akash Pawar
288,CUST0495,Priya Sharma


### 7. Products with More Returns than Purchases

In [10]:
query = '''
SELECT
    p.product_id,
    p.product_name,
    SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS purchases,
    SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END) AS returns
FROM products p
JOIN order_items oi
    ON p.product_id = oi.product_id
GROUP BY p.product_id, p.product_name
HAVING returns > purchases
ORDER BY returns DESC;
'''

display(pd.read_sql_query(query, conn))

,product_id,product_name,purchases,returns


### 8. Return Rate per Category

In [11]:
query = '''
SELECT
    p.category,
    SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END) AS returned_items,
    SUM(ABS(oi.quantity)) AS total_items,
    ROUND(
        100.0 * SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END)
        / NULLIF(SUM(ABS(oi.quantity)), 0),
        2
    ) AS return_rate_percent
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY return_rate_percent DESC;
'''

display(pd.read_sql_query(query, conn))

,category,returned_items,total_items,return_rate_percent
0,Clothing,59,1782,3.31
1,Electronics,61,1879,3.25
2,Books,65,2021,3.22
3,Home,47,1778,2.64


## Advanced Queries — Window Functions and CTEs

### 9. Running Total of Revenue per Region

In [12]:
query = '''
WITH daily AS (
    SELECT
        o.region_code,
        date(o.order_date) AS order_date,
        SUM(
            oi.quantity * oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS daily_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status NOT IN ('CANCELLED')
    GROUP BY o.region_code, date(o.order_date)
)
SELECT
    region_code,
    order_date,
    ROUND(daily_revenue, 2) AS daily_revenue,
    ROUND(
        SUM(daily_revenue) OVER (
            PARTITION BY region_code
            ORDER BY order_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS running_total
FROM daily
ORDER BY region_code, order_date;
'''

display(pd.read_sql_query(query, conn))

,region_code,order_date,daily_revenue,running_total
0,EAST,2025-01-05,355839.85,355839.85
1,EAST,2025-01-10,138218.44,494058.29
2,EAST,2025-01-19,119221.40,613279.69
3,EAST,2025-01-21,280280.92,893560.61
4,EAST,2025-01-23,81522.18,975082.79
...,...,...,...,...
610,WEST,2026-04-29,90325.19,24469429.72
611,WEST,2026-05-07,1026.42,24470456.14
612,WEST,2026-05-09,144143.36,24614599.49
613,WEST,2026-05-13,380789.76,24995389.25


### 10. Rank Products by Revenue within Category using DENSE_RANK

In [13]:
query = '''
WITH product_revenue AS (
    SELECT
        p.category,
        p.product_id,
        p.product_name,
        SUM(
            oi.quantity * oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS total_revenue
    FROM products p
    JOIN order_items oi ON p.product_id = oi.product_id
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status NOT IN ('CANCELLED')
    GROUP BY p.category, p.product_id, p.product_name
)
SELECT
    category,
    product_name,
    ROUND(total_revenue, 2) AS total_revenue,
    DENSE_RANK() OVER (
        PARTITION BY category
        ORDER BY total_revenue DESC
    ) AS rank_in_category
FROM product_revenue
ORDER BY category, rank_in_category;
'''

display(pd.read_sql_query(query, conn))

,category,product_name,total_revenue,rank_in_category
0,Books,Fiction Product 79,1840494.23,1
1,Books,Technology Product 89,1718065.68,2
2,Books,Comics Product 95,1683364.81,3
3,Books,Biography Product 99,1544873.44,4
4,Books,Biography Product 97,1537629.95,5
...,...,...,...,...
95,Home,Appliances Product 75,717217.77,21
96,Home,Decor Product 63,657494.99,22
97,Home,Appliances Product 72,646699.45,23
98,Home,Appliances Product 73,617838.02,24


### 11. LAG — Days Between Consecutive Customer Orders

In [14]:
query = '''
WITH customer_orders AS (
    SELECT
        customer_id,
        date(order_date) AS order_date,
        LAG(date(order_date)) OVER (
            PARTITION BY customer_id
            ORDER BY date(order_date)
        ) AS previous_order_date
    FROM orders
    WHERE customer_id IS NOT NULL
      AND customer_id <> 'UNKNOWN'
)
SELECT
    customer_id,
    order_date,
    previous_order_date,
    CASE
        WHEN previous_order_date IS NULL THEN NULL
        ELSE CAST(
            julianday(order_date) - julianday(previous_order_date)
            AS INTEGER
        )
    END AS days_gap
FROM customer_orders
ORDER BY customer_id, order_date;
'''

display(pd.read_sql_query(query, conn))

,customer_id,order_date,previous_order_date,days_gap
0,CUST0001,2025-02-19,None,NaN
1,CUST0001,2025-04-20,2025-02-19,60.0
2,CUST0001,2025-09-25,2025-04-20,158.0
3,CUST0002,2025-06-24,None,NaN
4,CUST0003,2025-01-20,None,NaN
...,...,...,...,...
947,CUST0498,2025-07-13,2025-03-14,121.0
948,CUST0499,2025-09-04,None,NaN
949,CUST0500,2025-03-22,None,NaN
950,CUST0500,2025-09-20,2025-03-22,182.0


### 12. Customers with Average Gap > 30 Days — At Risk

In [15]:
query = '''
WITH customer_orders AS (
    SELECT
        customer_id,
        date(order_date) AS order_date,
        LAG(date(order_date)) OVER (
            PARTITION BY customer_id
            ORDER BY date(order_date)
        ) AS previous_order_date
    FROM orders
    WHERE customer_id IS NOT NULL
      AND customer_id <> 'UNKNOWN'
),
gaps AS (
    SELECT
        customer_id,
        julianday(order_date) - julianday(previous_order_date) AS days_gap
    FROM customer_orders
    WHERE previous_order_date IS NOT NULL
)
SELECT
    customer_id,
    ROUND(AVG(days_gap), 2) AS average_gap_days,
    CASE
        WHEN AVG(days_gap) > 30 THEN 'At Risk'
        ELSE 'Active'
    END AS customer_status
FROM gaps
GROUP BY customer_id
ORDER BY average_gap_days DESC;
'''

display(pd.read_sql_query(query, conn))

,customer_id,average_gap_days,customer_status
0,CUST0149,479.0,At Risk
1,CUST0384,448.0,At Risk
2,CUST0295,443.0,At Risk
3,CUST0262,433.0,At Risk
4,CUST0355,420.0,At Risk
...,...,...,...
281,CUST0449,10.0,Active
282,CUST0470,7.0,Active
283,CUST0229,5.0,Active
284,CUST0078,3.0,Active


### 13. Multi-Level CTE — Monthly Customer Revenue Segmentation

In [16]:
query = '''
WITH monthly_revenue AS (
    SELECT
        o.customer_id,
        strftime('%Y-%m', o.order_date) AS month,
        SUM(
            oi.quantity * oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL
      AND o.customer_id <> 'UNKNOWN'
      AND o.status NOT IN ('CANCELLED')
    GROUP BY o.customer_id, strftime('%Y-%m', o.order_date)
),
categorized AS (
    SELECT
        customer_id,
        month,
        revenue,
        CASE
            WHEN revenue > 10000 THEN 'High'
            WHEN revenue >= 5000 THEN 'Medium'
            ELSE 'Low'
        END AS revenue_category
    FROM monthly_revenue
)
SELECT
    month,
    revenue_category,
    COUNT(*) AS customer_count
FROM categorized
GROUP BY month, revenue_category
ORDER BY month, revenue_category;
'''

display(pd.read_sql_query(query, conn))

,month,revenue_category,customer_count
0,2025-01,High,37
1,2025-01,Low,1
2,2025-02,High,32
3,2025-02,Low,3
4,2025-03,High,36
5,2025-03,Low,4
6,2025-03,Medium,1
7,2025-04,High,29
8,2025-04,Low,1
9,2025-04,Medium,2


### 14. NTILE — Lifetime Value Quartiles

In [17]:
query = '''
WITH lifetime_value AS (
    SELECT
        o.customer_id,
        SUM(
            oi.quantity * oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS total_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL
      AND o.customer_id <> 'UNKNOWN'
      AND o.status NOT IN ('CANCELLED')
    GROUP BY o.customer_id
),
quartiles AS (
    SELECT
        customer_id,
        total_value,
        NTILE(4) OVER (
            ORDER BY total_value DESC
        ) AS quartile
    FROM lifetime_value
)
SELECT
    customer_id,
    ROUND(total_value, 2) AS total_value,
    quartile,
    CASE quartile
        WHEN 1 THEN 'Platinum'
        WHEN 2 THEN 'Gold'
        WHEN 3 THEN 'Silver'
        WHEN 4 THEN 'Bronze'
    END AS quartile_label
FROM quartiles
ORDER BY quartile, total_value DESC;
'''

display(pd.read_sql_query(query, conn))

,customer_id,total_value,quartile,quartile_label
0,CUST0497,1040001.00,1,Platinum
1,CUST0425,1034103.69,1,Platinum
2,CUST0148,974070.73,1,Platinum
3,CUST0372,929067.63,1,Platinum
4,CUST0308,889125.72,1,Platinum
...,...,...,...,...
376,CUST0446,-9641.54,4,Bronze
377,CUST0041,-15915.06,4,Bronze
378,CUST0334,-17298.32,4,Bronze
379,CUST0226,-19361.37,4,Bronze


### 15. Year-over-Year Revenue Comparison

In [18]:
query = '''
WITH monthly AS (
    SELECT
        strftime('%Y', o.order_date) AS year,
        strftime('%m', o.order_date) AS month,
        SUM(
            oi.quantity * oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status NOT IN ('CANCELLED')
    GROUP BY strftime('%Y', o.order_date),
             strftime('%m', o.order_date)
),
with_previous AS (
    SELECT
        year,
        month,
        revenue,
        LAG(revenue, 12) OVER (
            ORDER BY year, month
        ) AS prev_year_revenue
    FROM monthly
)
SELECT
    year,
    month,
    ROUND(revenue, 2) AS revenue,
    ROUND(prev_year_revenue, 2) AS prev_year_revenue,
    CASE
        WHEN prev_year_revenue IS NULL THEN NULL
        WHEN prev_year_revenue = 0 THEN NULL
        ELSE ROUND(
            100.0 * (revenue - prev_year_revenue)
            / prev_year_revenue,
            2
        )
    END AS yoy_growth_percent
FROM with_previous
ORDER BY year, month;
'''

display(pd.read_sql_query(query, conn))

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2025,01,5459334.05,NaN,NaN
1,2025,02,5889118.13,NaN,NaN
2,2025,03,5942908.94,NaN,NaN
3,2025,04,3628871.58,NaN,NaN
4,2025,05,6108831.32,NaN,NaN
5,2025,06,8073138.76,NaN,NaN
6,2025,07,6990208.74,NaN,NaN
7,2025,08,7000303.28,NaN,NaN
8,2025,09,6399816.19,NaN,NaN
9,2025,10,6360471.80,NaN,NaN


### 16. First and Most Recent Purchased Category

In [19]:
query = '''
WITH customer_category_orders AS (
    SELECT
        o.customer_id,
        p.category,
        o.order_date,
        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date
        ) AS first_rank,
        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date DESC
        ) AS last_rank
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    WHERE o.customer_id IS NOT NULL
      AND o.customer_id <> 'UNKNOWN'
),
first_last AS (
    SELECT
        customer_id,
        MAX(CASE WHEN first_rank = 1 THEN category END) AS first_category,
        MAX(CASE WHEN last_rank = 1 THEN category END) AS most_recent_category
    FROM customer_category_orders
    GROUP BY customer_id
)
SELECT
    customer_id,
    first_category,
    most_recent_category,
    CASE
        WHEN first_category <> most_recent_category THEN 'Yes'
        ELSE 'No'
    END AS category_shift
FROM first_last
ORDER BY customer_id;
'''

display(pd.read_sql_query(query, conn))

,customer_id,first_category,most_recent_category,category_shift
0,CUST0001,Clothing,Home,Yes
1,CUST0002,Electronics,Electronics,No
2,CUST0003,Electronics,Books,Yes
3,CUST0004,Electronics,Electronics,No
4,CUST0005,Electronics,Electronics,No
...,...,...,...,...
422,CUST0495,Electronics,Electronics,No
423,CUST0497,Electronics,Home,Yes
424,CUST0498,Electronics,Clothing,Yes
425,CUST0499,Electronics,Electronics,No


### 17. Cumulative Distribution — Revenue Contribution by Customer

In [20]:
query = '''
WITH customer_revenue AS (
    SELECT
        o.customer_id,
        SUM(
            oi.quantity * oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL
      AND o.customer_id <> 'UNKNOWN'
      AND o.status NOT IN ('CANCELLED')
    GROUP BY o.customer_id
),
ranked AS (
    SELECT
        customer_id,
        revenue,
        SUM(revenue) OVER (
            ORDER BY revenue DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_revenue,
        SUM(revenue) OVER () AS total_revenue
    FROM customer_revenue
)
SELECT
    customer_id,
    ROUND(revenue, 2) AS revenue,
    ROUND(cumulative_revenue, 2) AS cumulative_revenue,
    ROUND(
        100.0 * cumulative_revenue / NULLIF(total_revenue, 0),
        2
    ) AS cumulative_percent
FROM ranked
ORDER BY revenue DESC;
'''

display(pd.read_sql_query(query, conn))

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,CUST0497,1040001.00,1040001.00,1.07
1,CUST0425,1034103.69,2074104.69,2.14
2,CUST0148,974070.73,3048175.42,3.14
3,CUST0372,929067.63,3977243.05,4.10
4,CUST0308,889125.72,4866368.77,5.02
...,...,...,...,...
376,CUST0446,-9641.54,97072218.28,100.08
377,CUST0041,-15915.06,97056303.22,100.06
378,CUST0334,-17298.32,97039004.90,100.04
379,CUST0226,-19361.37,97019643.53,100.02


### 18. Cohort Analysis — Registration Month and Retention

In [21]:
query = '''
WITH customer_cohort AS (
    SELECT
        customer_id,
        strftime('%Y-%m', registration_date) AS cohort_month
    FROM customers
),
customer_orders AS (
    SELECT DISTINCT
        o.customer_id,
        strftime('%Y-%m', o.order_date) AS order_month
    FROM orders o
    WHERE o.customer_id IS NOT NULL
      AND o.customer_id <> 'UNKNOWN'
),
cohort_activity AS (
    SELECT
        c.cohort_month,
        c.customer_id,
        CAST(
            (
                (CAST(substr(o.order_month, 1, 4) AS INTEGER) -
                 CAST(substr(c.cohort_month, 1, 4) AS INTEGER)) * 12
                +
                (CAST(substr(o.order_month, 6, 2) AS INTEGER) -
                 CAST(substr(c.cohort_month, 6, 2) AS INTEGER))
            ) AS INTEGER
        ) AS month_number
    FROM customer_cohort c
    JOIN customer_orders o
        ON c.customer_id = o.customer_id
    WHERE month_number BETWEEN 0 AND 3
),
cohort_sizes AS (
    SELECT
        cohort_month,
        COUNT(*) AS cohort_size
    FROM customer_cohort
    GROUP BY cohort_month
),
retention AS (
    SELECT
        cohort_month,
        month_number,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM cohort_activity
    GROUP BY cohort_month, month_number
)
SELECT
    r.cohort_month,
    cs.cohort_size,
    r.month_number,
    r.active_customers,
    ROUND(
        100.0 * r.active_customers / cs.cohort_size,
        2
    ) AS retention_rate_percent
FROM retention r
JOIN cohort_sizes cs
    ON r.cohort_month = cs.cohort_month
ORDER BY r.cohort_month, r.month_number;
'''

display(pd.read_sql_query(query, conn))

,cohort_month,cohort_size,month_number,active_customers,retention_rate_percent
0,2024-11,8,3,3,37.50
1,2024-12,17,2,1,5.88
2,2024-12,17,3,3,17.65
3,2025-01,20,0,2,10.00
4,2025-01,20,1,3,15.00
5,2025-01,20,2,1,5.00
6,2025-01,20,3,3,15.00
7,2025-02,17,0,1,5.88
8,2025-02,17,1,3,17.65
9,2025-02,17,2,1,5.88


In [26]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("your_database.db")

print("Database connected successfully")

Database connected successfully


### 19. Self-Join with Window Function — Frequently Bought Together

In [28]:
query = '''
WITH item_pairs AS (
    SELECT
        oi1.order_id,
        CASE
            WHEN oi1.product_id < oi2.product_id
            THEN oi1.product_id
            ELSE oi2.product_id
        END AS product_a,
        CASE
            WHEN oi1.product_id < oi2.product_id
            THEN oi2.product_id
            ELSE oi1.product_id
        END AS product_b
    FROM order_items oi1
    JOIN order_items oi2
        ON oi1.order_id = oi2.order_id
       AND oi1.product_id < oi2.product_id
),
pair_counts AS (
    SELECT
        product_a,
        product_b,
        COUNT(DISTINCT order_id) AS times_bought_together
    FROM item_pairs
    GROUP BY product_a, product_b
)
SELECT
    pc.product_a,
    pa.product_name AS product_a_name,
    pc.product_b,
    pb.product_name AS product_b_name,
    pc.times_bought_together
FROM pair_counts pc
JOIN products pa ON pc.product_a = pa.product_id
JOIN products pb ON pc.product_b = pb.product_id
ORDER BY pc.times_bought_together DESC;
'''

result = pd.read_sql_query(query, conn)
display(result)

DatabaseError: Execution failed on sql '
WITH item_pairs AS (
    SELECT
        oi1.order_id,
        CASE
            WHEN oi1.product_id < oi2.product_id
            THEN oi1.product_id
            ELSE oi2.product_id
        END AS product_a,
        CASE
            WHEN oi1.product_id < oi2.product_id
            THEN oi2.product_id
            ELSE oi1.product_id
        END AS product_b
    FROM order_items oi1
    JOIN order_items oi2
        ON oi1.order_id = oi2.order_id
       AND oi1.product_id < oi2.product_id
),
pair_counts AS (
    SELECT
        product_a,
        product_b,
        COUNT(DISTINCT order_id) AS times_bought_together
    FROM item_pairs
    GROUP BY product_a, product_b
)
SELECT
    pc.product_a,
    pa.product_name AS product_a_name,
    pc.product_b,
    pb.product_name AS product_b_name,
    pc.times_bought_together
FROM pair_counts pc
JOIN products pa ON pc.product_a = pa.product_id
JOIN products pb ON pc.product_b = pb.product_id
ORDER BY pc.times_b

## 20. Close the Database Connection

In [24]:
conn.close()
print("SQLite connection closed.")

SQLite connection closed.
